# 🧠 Brain Tumor MRI — Global Cross-Dataset Duplicate Detector
### Perceptual Hash (pHash) | Priority-Based Deletion | N-Way Leakage Matrix

---
**Strategy:**  
1. Compute `pHash` for every image in all datasets concurrently  
2. Group images by hash — any group with images from **multiple datasets** is a cross-contamination hit  
3. Keep the copy in the **lowest priority number** (highest importance), delete the rest  
4. Report a full **Leakage Matrix** showing which dataset lost how many images to whom

> ⚠️ **Permanent deletion is active.** Duplicates are removed with `os.remove()`. No undo.

## ⚙️ Step 0 — Install Dependencies

In [55]:
%pip install imagehash 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.7 MB/s eta 0:00:00


## 📦 Step 1 — Imports

In [56]:
import os
import sys
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import imagehash
from PIL import Image
from tqdm.auto import tqdm

print(f"✅ imagehash v{imagehash.__version__} loaded")
print(f"✅ Python {sys.version.split()[0]}")

✅ imagehash v4.3.2 loaded
✅ Python 3.12.13


In [57]:

from google.colab import drive
drive.mount("/content/drive")
DRIVE_PATH = Path("/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!find "{DRIVE_PATH}" -maxdepth 3 -type d

/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1/test
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1/train
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2/test
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2/train
/

In [58]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================================
# 1. SETUP & CONFIGURATION
# =====================================================================

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}
MAX_WORKERS = 8

# Grouped dictionary: Priority 1 = Keep, Priority 5 = Delete.
DATASETS = {
    "Source_BRISC":    {"path": DRIVE_PATH / "DANN/source", "priority": 1},
    "Target_Mendeley": {"path": DRIVE_PATH / "DANN/target", "priority": 2},
    "Val_Ayesha":      {"path": DRIVE_PATH / "External Validation/External-validation-dataset-1", "priority": 3},
    "Val_Alam":        {"path": DRIVE_PATH / "External Validation/External-validation-dataset-2", "priority": 4},
    "Val_DeepPy":      {"path": DRIVE_PATH / "External Validation/External-validation-dataset-3", "priority": 5},
}

# Sort datasets by priority for consistent output
DATASETS_BY_PRIORITY = sorted(DATASETS.items(), key=lambda x: x[1]["priority"])


# =====================================================================
# 2. PATH VALIDATION & PRIORITY CHECK
# =====================================================================
print("Priority order (1=keep, 5=delete):")
for name, cfg in DATASETS_BY_PRIORITY:
    exists = cfg["path"].exists()
    status = "✅" if exists else "❌ PATH NOT FOUND"
    print(f"  [{cfg['priority']}] {name:<20} → {cfg['path']}  {status}")


# =====================================================================
# 3. UTILITIES
# =====================================================================
def count_images(folder_path):
    """Recursively counts images in the given directory using the global VALID_EXTENSIONS."""
    directory = Path(folder_path)
    
    if not directory.exists():
        return "Folder not found"
        
    return sum(
        1 for file in directory.rglob('*') 
        if file.is_file() and file.suffix.lower() in VALID_EXTENSIONS
    )


# =====================================================================
# 4. EXECUTION & REPORTING
# =====================================================================
print(f"\n{'Dataset Name':<20} | {'Train Images':<15} | {'Test Images'}")
print("-" * 60)

# Iterate through the sorted list to keep the output in order of priority
for name, cfg in DATASETS_BY_PRIORITY:
    base_path = cfg["path"]
    
    # Path library makes joining folders as simple as using the '/' operator
    train_count = count_images(base_path / "train")
    test_count = count_images(base_path / "test")
    
    print(f"{name:<20} | {str(train_count):<15} | {str(test_count)}")

Priority order (1=keep, 5=delete):
  [1] Source_BRISC         → /content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/DANN/source  ✅
  [2] Target_Mendeley      → /content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/DANN/target  ✅
  [3] Val_Ayesha           → /content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1  ✅
  [4] Val_Alam             → /content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-2  ✅
  [5] Val_DeepPy           → /content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-3  ✅

Dataset Name         | Train Images    | Test Images
------------------------------------------------------------
Source_BRISC         | 5000            | 1000
Target_Mendeley      | 9658            | 2414
Val_Ayesha    

In [66]:
def collect_images(dataset_name: str, root_path: str):
    root = Path(root_path)
    if not root.exists():
        return []
    return [(dataset_name, str(p.resolve())) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS]

all_images = []
dataset_image_counts = {}

for name, cfg in DATASETS_BY_PRIORITY:
    imgs = collect_images(name, cfg["path"])
    dataset_image_counts[name] = len(imgs)
    all_images.extend(imgs)
    print(f"  {name:<20}: {len(imgs):>5} images found")

print(f"\n📁 Total images to hash: {len(all_images):,}")

  Source_BRISC        :  6000 images found
  Target_Mendeley     :  8015 images found
  Val_Ayesha          :  1423 images found
  Val_Alam            :  5678 images found
  Val_DeepPy          :  1243 images found

📁 Total images to hash: 22,359


In [67]:
def compute_phash(filepath):
    try:
        with Image.open(filepath) as img:
            return str(imagehash.phash(img)) # Convert to string for stable matching
    except Exception:
        return None

hash_map = defaultdict(list)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(compute_phash, img[1]): img for img in all_images}
    
    for future in tqdm(as_completed(futures), total=len(all_images), desc="pHashing"):
        dataset_name, filepath = futures[future]
        h = future.result()
        if h is not None:
            hash_map[h].append((dataset_name, filepath))

print(f"✅ Hashing complete. Unique visual signatures found: {len(hash_map)}")

pHashing:   0%|          | 0/22359 [00:00<?, ?it/s]

✅ Hashing complete. Unique visual signatures found: 14405


In [68]:
leakage_matrix = defaultdict(lambda: defaultdict(int))
leakage_pairs = []       # Cross-dataset duplicates
internal_pairs = []      # Same-folder duplicates
internal_totals = defaultdict(int)

for hash_str, entries in hash_map.items():
    # --- 1. HANDLE INTERNAL DUPLICATES FIRST ---
    # Group entries by dataset to find duplicates sitting in the same folder
    ds_to_paths = defaultdict(list)
    for ds, path in entries:
        ds_to_paths[ds].append(path)
        
    unique_entries = [] 
    
    for ds, paths in ds_to_paths.items():
        # Keep the very first path we find for this specific dataset
        unique_entries.append((ds, paths[0]))
        
        # Any extra paths in the exact same dataset are internal duplicates
        for dup_path in paths[1:]:
            internal_pairs.append((ds, dup_path))
            internal_totals[ds] += 1
            
    # --- 2. HANDLE CROSS-DATASET LEAKAGE ---
    # If this hash only exists in one dataset (after internal cleanup), skip
    if len(unique_entries) < 2:
        continue 

    # Sort so the lowest priority number (highest importance) is first
    sorted_entries = sorted(unique_entries, key=lambda e: DATASETS[e[0]]["priority"])
    keeper_ds, keeper_path = sorted_entries[0]

    seen_datasets = {keeper_ds}
    for dup_ds, dup_path in sorted_entries[1:]:
        if dup_ds not in seen_datasets:
            leakage_pairs.append((keeper_ds, keeper_path, dup_ds, dup_path))
            leakage_matrix[keeper_ds][dup_ds] += 1
            seen_datasets.add(dup_ds)

print(f"🗑️  Internal duplicates found: {len(internal_pairs)}")
print(f"🔎 Cross-dataset duplicates found: {len(leakage_pairs)}")

🗑️  Internal duplicates found: 3543
🔎 Cross-dataset duplicates found: 4411


In [69]:
actual_deleted_internal = 0
actual_deleted_cross = 0

# 1. Delete Internal Duplicates
if internal_pairs:
    print(f"\n🧹 Starting deletion of {len(internal_pairs)} INTERNAL files...")
    for ds, del_path in tqdm(internal_pairs, desc="Internal Dels"):
        try:
            if os.path.exists(del_path):
                os.remove(del_path)
                actual_deleted_internal += 1
        except Exception as e:
            pass

# 2. Delete Cross-Dataset Duplicates
if leakage_pairs:
    print(f"\n⚠️  Starting deletion of {len(leakage_pairs)} CROSS-DATASET files...")
    for _, _, _, del_path in tqdm(leakage_pairs, desc="External Dels"):
        try:
            if os.path.exists(del_path):
                os.remove(del_path)
                actual_deleted_cross += 1
        except Exception as e:
            pass

print(f"\n✅ Successfully removed {actual_deleted_internal} internal copies.")
print(f"✅ Successfully removed {actual_deleted_cross} cross-dataset leaks.")


🧹 Starting deletion of 3543 INTERNAL files...


Internal Dels:   0%|          | 0/3543 [00:00<?, ?it/s]


⚠️  Starting deletion of 4411 CROSS-DATASET files...


External Dels:   0%|          | 0/4411 [00:00<?, ?it/s]


✅ Successfully removed 3543 internal copies.
✅ Successfully removed 4411 cross-dataset leaks.


In [70]:
COL_W, LABEL_W = 18, 20 

print("\n" + "═" * (LABEL_W + COL_W * len(dataset_names) + 4))
print(" 🔬 CROSS-DATASET LEAKAGE MATRIX")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 4))
print("\n  Rows = SOURCE (kept)   │   Cols = TARGET (deleted from)\n")

header = f"{'SOURCE \\ TARGET':<{LABEL_W}}"
for col in dataset_names:
    p = DATASETS[col]["priority"]
    header += f"[P{p}]{col:<{COL_W - 4}}"
print(header)
print("─" * len(header))

col_totals = defaultdict(int)
for row_name in dataset_names:
    p = DATASETS[row_name]["priority"]
    row_str = f"[P{p}]{row_name:<{LABEL_W - 4}}"
    row_sum = 0
    for col_name in dataset_names:
        if col_name == row_name:
            row_str += f"{'—':>{COL_W}}"
        else:
            count = leakage_matrix[row_name].get(col_name, 0)
            row_str += f"{str(count) if count > 0 else '0':>{COL_W}}"
            row_sum += count
            col_totals[col_name] += count
    print(row_str + f"  │ Total external deleted: {row_sum}")

print("─" * len(header))
totals_row = f"{'Col Total (deleted)':<{LABEL_W}}"
for col_name in dataset_names:
    totals_row += f"{col_totals.get(col_name, 0):>{COL_W}}"
print(totals_row)

print("\n" + "═" * (LABEL_W + COL_W * len(dataset_names) + 40))
print(" 📋 GLOBAL DEDUPLICATION SUMMARY")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 40))

for name in dataset_names:
    p = DATASETS[name]["priority"]
    original_count = dataset_image_counts.get(name, 0)
    
    internal_deleted = internal_totals.get(name, 0)
    external_deleted = col_totals.get(name, 0)   
    remaining = original_count - internal_deleted - external_deleted
    
    print(f"  [P{p}] {name:<20} | Original: {original_count:>5} | "
          f"Internal Dels: {internal_deleted:>5} | "
          f"External Dels: {external_deleted:>5} | "
          f"Remaining: {remaining:>5}")

print(f"\n  {'Total Internal Copies Removed':.<40} {actual_deleted_internal}")
print(f"  {'Total Cross-Dataset Leaks Removed':.<40} {actual_deleted_cross}")
print(f"  {'GRAND TOTAL DELETED':.<40} {actual_deleted_internal + actual_deleted_cross}")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 40))


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 🔬 CROSS-DATASET LEAKAGE MATRIX
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Rows = SOURCE (kept)   │   Cols = TARGET (deleted from)

SOURCE \ TARGET     [P1]Source_BRISC  [P2]Target_Mendeley[P3]Val_Ayesha    [P4]Val_Alam      [P5]Val_DeepPy    
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
[P1]Source_BRISC                     —              1821               141              1099               249  │ Total external deleted: 3310
[P2]Target_Mendeley                  0                 —               439               172               439  │ Total external deleted: 1050
[P3]Val_Ayesha                       0                 0                 —                12                39  │ Total external deleted: 51
[P4]Val_Alam              